## **Google Maps - Turisim Type**

In [ ]:
from openpyxl import load_workbook
import pandas as pd
import glob
import os
import re

# المسار الأساسي لبيانات Google Maps
input_path = r"C:\Users\ziyad\OneDrive\Desktop\tourism type\Cleaned Data From Google Maps"

# مسار ملف الإخراج Excel
output_path = r"C:\Users\ziyad\OneDrive\Desktop\tourism type\Google_Maps_Tourism_Classification_Result1.xlsx"

# قراءة كل ملفات Excel داخل المجلد والمجلدات الفرعية
files = glob.glob(input_path + r"\**\*.xlsx", recursive=True)

tourism_keywords = {
    "Cultural Tourism": ['متحف', 'مكتبة', 'تراث', 'ثقافي', 'معرض' , 'الحرف اليدوية'],
    "Entertainment Tourism": ['حديقة', 'بوليفارد', 'ملاهي', 'فعاليات', 'ترفيه', 'شاليهات', 'منتجع', 'مهرجان'],
    "Historical Tourism": ['قلعة', 'آثار', 'اثار', 'تاريخي', 'قصر', 'حصن' , 'قرية'],
    "Religious Tourism": ['مسجد', 'جامع', 'الحرم', 'نبوي', 'الكعبة'],
    "Nature Tourism": ['جبل', 'غابة', 'شلال', 'وادي', 'شاطئ', 'بحر', 'بحيرة', 'منتزه', 'أكواخ', 'جزيرة', 'مزرعة', 'محمية', 'موانئ' , 'ميناء' ,'مطل']
}

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def classify_place(category, title):

    category = clean_text(category)
    title = clean_text(title)

    best_type = "Other"
    score = 0
    matched_keywords = []
    classification_source = []

    for tourism_type, keywords in tourism_keywords.items():
        temp_score = 0
        temp_keywords = []
        temp_sources = []

        for word in keywords:
            if word in category:
                temp_score += 0.50
                temp_keywords.append(word)
                temp_sources.append("categoryName")

            if word in title:
                temp_score += 0.20
                temp_keywords.append(word)
                temp_sources.append("title")

        if temp_score > score:
            score = round(min(temp_score, 1.0), 2)
            best_type = tourism_type
            matched_keywords = list(set(temp_keywords))
            classification_source = list(set(temp_sources))

    return best_type, score, matched_keywords, classification_source

results = []

for file in files:

    try:
        wb = load_workbook(file, read_only=True, data_only=True)
        ws = wb.active

        rows = ws.iter_rows(values_only=True)

        header = next(rows, None)
        first_row = next(rows, None)

        if header is None or first_row is None:
            wb.close()
            continue

        row_dict = dict(zip(header, first_row))

        category = row_dict.get('categoryName', '')
        title = row_dict.get('title', '')

        tourism_type, score, matched_keywords, classification_source = classify_place(
            category,
            title
        )

        results.append({
            "Place_Name": title,
            "CategoryName": category,
            "Tourism_Type": tourism_type,
            "Confidence_Score": score,
            "Matched_Keywords": ", ".join(matched_keywords),
            "Classification_Source": ", ".join(classification_source),
            "Source_File": os.path.basename(file)
        })

        wb.close()

    except Exception as e:
        print(f"Error in file: {file}")
        print(e)

df = pd.DataFrame(results)

# حفظ النتائج بصيغة Excel
df.to_excel(output_path, index=False)

print(df.head(10))
print(f"\nTotal Excel Files Found: {len(files)}")
print(f"Total Classified Places: {len(df)}")
print(f"\nSaved Excel File: {output_path}")

                       Place_Name         CategoryName           Tourism_Type  \
0               اكواخ سار الريفية           فندق منتجع  Entertainment Tourism   
1        القرية الأثرية بالأطاولة  المحافظة على التراث       Cultural Tourism   
2                  جبل شدا الأعلى              قمة جبل         Nature Tourism   
3     حديقة الأمير سلطان بن سلمان                حديقة  Entertainment Tourism   
4       حديقة الأمير محمد بن سعود                حديقة  Entertainment Tourism   
5            حديقة الجسر النباتية                حديقة  Entertainment Tourism   
6       حديقة الحسام (حديقة شهبة)                حديقة  Entertainment Tourism   
7  حديقه الخزامى ( خياصه سابقاً )                حديقة  Entertainment Tourism   
8                    حديقة الشمال                حديقة  Entertainment Tourism   
9            حديقه الغرير بالمندق                حديقة  Entertainment Tourism   

   Confidence_Score Matched_Keywords Classification_Source  \
0               0.5            منتجع          

## **Tik Tok - Turisim Type**

In [11]:
# ============================================================
# Add Tourism_Type to ALL Google Maps Excel Files
# Supports:
# categoryName / CategoryName
# title / Place Name
# ============================================================

import pandas as pd
import glob
import os
import re
import traceback

# ============================================================
# 1. Paths
# ============================================================

input_path = r"C:\Users\ziyad\OneDrive\Desktop\tourism type\Cleaned Data From Google Maps"

output_path = r"C:\Users\ziyad\OneDrive\Desktop\tourism type\Google Maps With Tourism Type Final2"

summary_output = r"C:\Users\ziyad\OneDrive\Desktop\tourism type\Tourism_Type_Files_Summary_Final.xlsx"

error_log_output = r"C:\Users\ziyad\OneDrive\Desktop\tourism type\Tourism_Type_Error_Log_Final.xlsx"

os.makedirs(output_path, exist_ok=True)

# ============================================================
# 2. Tourism Keywords Dictionary
# ============================================================

tourism_keywords = {

    "Cultural Tourism": [
        'متحف', 'مكتبة', 'تراث', 'ثقافي', 'معرض',
        'الحرف اليدوية', 'فنون', 'ثقافة', 'حضاري', 'تراثي'
    ],

    "Entertainment Tourism": [
        'حديقة', 'بوليفارد', 'ملاهي', 'فعاليات',
        'ترفيه', 'شاليهات', 'منتجع', 'مهرجان',
        'العاب', 'ألعاب'
    ],

    "Historical Tourism": [
        'قلعة', 'آثار', 'اثار', 'تاريخي',
        'قصر', 'حصن', 'قرية', 'أثري',
        'اثرية', 'تراثية'
    ],

    "Religious Tourism": [
        'مسجد', 'جامع', 'الحرم', 'نبوي',
        'الكعبة', 'مصلى', 'ديني', 'اسلامي', 'إسلامي'
    ],

    "Nature Tourism": [
        'جبل', 'غابة', 'شلال', 'وادي',
        'شاطئ', 'بحر', 'بحيرة', 'منتزه',
        'أكواخ', 'اكواخ', 'جزيرة', 'مزرعة',
        'محمية', 'موانئ', 'ميناء', 'مطل'
    ]
}

# ============================================================
# 3. Helper Functions
# ============================================================

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def normalize_col_name(col):
    return str(col).strip().lower().replace(" ", "").replace("_", "")


def find_column(df, possible_names):
    normalized_columns = {
        normalize_col_name(col): col
        for col in df.columns
    }

    for name in possible_names:
        key = normalize_col_name(name)
        if key in normalized_columns:
            return normalized_columns[key]

    return None


def classify_place(category, place_name, file_name):

    category = clean_text(category)
    place_name = clean_text(place_name)
    file_name = clean_text(file_name)

    best_type = "Other"
    best_score = 0

    for tourism_type, keywords in tourism_keywords.items():

        score = 0

        for word in keywords:
            word = clean_text(word)

            if word in category:
                score += 0.50

            if word in place_name:
                score += 0.30

            if word in file_name:
                score += 0.20

        if score > best_score:
            best_score = score
            best_type = tourism_type

    return best_type, round(min(best_score, 1.0), 2)

# ============================================================
# 4. Collect Excel Files
# ============================================================

files = glob.glob(input_path + r"\**\*.xlsx", recursive=True)

files = [
    f for f in files
    if not os.path.basename(f).startswith("~$")
]

print(f"Total Excel files found: {len(files)}")

# ============================================================
# 5. Process All Files
# ============================================================

summary = []
errors = []

for i, file in enumerate(files, start=1):

    print(f"\nProcessing {i}/{len(files)}")
    print(file)

    try:
        df = pd.read_excel(file, engine="openpyxl")

        if df.empty:
            errors.append({
                "File": file,
                "Error": "Empty file"
            })
            print("Skipped: Empty file")
            continue

        # الأعمدة الفعلية الموجودة عندكم
        category_col = find_column(df, [
            "categoryName",
            "CategoryName"
        ])

        place_col = find_column(df, [
            "title",
            "Title",
            "Place Name",
            "PlaceName",
            "Place_Name"
        ])

        if category_col is None or place_col is None:
            errors.append({
                "File": file,
                "Error": "Missing required columns",
                "Available_Columns": ", ".join([str(c) for c in df.columns])
            })
            print("Skipped: Missing required columns")
            print("Available columns:", list(df.columns))
            continue

        category_series = df[category_col].dropna()
        place_series = df[place_col].dropna()

        category = category_series.iloc[0] if len(category_series) > 0 else ""
        place_name = place_series.iloc[0] if len(place_series) > 0 else ""

        file_name = os.path.basename(file)

        tourism_type, confidence_score = classify_place(
            category,
            place_name,
            file_name
        )

        # حذف الأعمدة القديمة لو موجودة
        old_columns = [
            "نوع السياحة",
            "درجة الثقة",
            "Tourism_Type",
            "Confidence_Score"
        ]

        for col in old_columns:
            if col in df.columns:
                df = df.drop(columns=[col])

        # إضافة الأعمدة الإنجليزية
        df["Tourism_Type"] = tourism_type
        df["Confidence_Score"] = confidence_score

        # حفظ بنفس هيكلة المجلدات
        relative_path = os.path.relpath(file, input_path)
        new_file_path = os.path.join(output_path, relative_path)

        os.makedirs(os.path.dirname(new_file_path), exist_ok=True)

        df.to_excel(new_file_path, index=False, engine="openpyxl")

        summary.append({
            "File_Name": file_name,
            "Place_Name": place_name,
            "CategoryName": category,
            "Used_Category_Column": category_col,
            "Used_Place_Column": place_col,
            "Tourism_Type": tourism_type,
            "Confidence_Score": confidence_score,
            "Saved_To": new_file_path
        })

        print(f"Saved: {file_name} -> {tourism_type}")

    except Exception as e:
        errors.append({
            "File": file,
            "Error": str(e),
            "Traceback": traceback.format_exc()
        })

        print("ERROR:")
        print(e)

# ============================================================
# 6. Save Summary and Error Log
# ============================================================

summary_df = pd.DataFrame(summary)
errors_df = pd.DataFrame(errors)

summary_df.to_excel(summary_output, index=False, engine="openpyxl")
errors_df.to_excel(error_log_output, index=False, engine="openpyxl")

print("\nDONE")
print(f"Processed files: {len(summary_df)}")
print(f"Error files: {len(errors_df)}")
print(f"Summary saved at: {summary_output}")
print(f"Error log saved at: {error_log_output}")

Total Excel files found: 442

Processing 1/442
C:\Users\ziyad\OneDrive\Desktop\tourism type\Cleaned Data From Google Maps\( المنطقة الجنوبية ) Google Maps Data -  After Cleaning\منطقة الباحة\أكواخ سار الريفية_textready.xlsx
Saved: أكواخ سار الريفية_textready.xlsx -> Entertainment Tourism

Processing 2/442
C:\Users\ziyad\OneDrive\Desktop\tourism type\Cleaned Data From Google Maps\( المنطقة الجنوبية ) Google Maps Data -  After Cleaning\منطقة الباحة\القرية الأثرية بالأطاولة_textready.xlsx
Saved: القرية الأثرية بالأطاولة_textready.xlsx -> Historical Tourism

Processing 3/442
C:\Users\ziyad\OneDrive\Desktop\tourism type\Cleaned Data From Google Maps\( المنطقة الجنوبية ) Google Maps Data -  After Cleaning\منطقة الباحة\جبل شدا الأعلى_textready.xlsx
Saved: جبل شدا الأعلى_textready.xlsx -> Nature Tourism

Processing 4/442
C:\Users\ziyad\OneDrive\Desktop\tourism type\Cleaned Data From Google Maps\( المنطقة الجنوبية ) Google Maps Data -  After Cleaning\منطقة الباحة\حديقة الأمير سلطان بن سلمان_tex

In [1]:
# ============================================================
# TikTok Tourism Classification Pipeline
# Row-Level Tourism Classification
# ============================================================

import pandas as pd
import glob
import os
import re
import traceback

# ============================================================
# 1. Paths
# ============================================================

# مسار بيانات TikTok
input_path = r"C:\Users\aws12\Desktop\Create Turism Type\Tik Tok Datasets - After Processing"

# مسار حفظ النتائج
output_path = r"C:\Users\aws12\Desktop\Create Turism Type\OutPut Result"

# ملفات الملخص والأخطاء
summary_output = r"C:\Users\aws12\Desktop\Create Turism Type\summary output"

error_output = r"C:\Users\aws12\Desktop\Create Turism Type\error output"

os.makedirs(output_path, exist_ok=True)

# ============================================================
# 2. Tourism Keywords
# ============================================================

tourism_keywords = {

    "Cultural Tourism": [
        'تراث', 'ثقافة', 'ثقافي', 'متحف',
        'حضاري', 'الحرف اليدوية', 'فنون',
        'تراثي'
    ],

    "Entertainment Tourism": [
        'فعاليات', 'ترفيه', 'مهرجان',
        'بوليفارد', 'حفلات', 'حفلة',
        'العاب', 'ألعاب', 'موسم',
        'ونتر', 'سيتي ووك'
    ],

    "Historical Tourism": [
        'آثار', 'اثار', 'تاريخ', 'تاريخي',
        'قلعة', 'قصر', 'حصن',
        'قرية تاريخية', 'اثرية'
    ],

    "Religious Tourism": [
        'الحرم', 'الكعبة', 'المسجد النبوي',
        'مكة', 'المدينة', 'عمرة',
        'حج', 'اسلامي', 'إسلامي'
    ],

    "Nature Tourism": [
        'طبيعة', 'جبال', 'جبل',
        'غابة', 'شلال', 'وادي',
        'بحر', 'شاطئ', 'ضباب',
        'مطر', 'أجواء', 'مزرعة',
        'منتزه', 'غيم', 'سحاب'
    ]
}

# ============================================================
# 3. Text Cleaning
# ============================================================

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# ============================================================
# 4. Detect Text Column
# ============================================================

def detect_text_column(df):

    possible_columns = [
        "Text",
        "text",
        "Text_TR",
        "Text_ML",
        "comment",
        "Comment",
        "content",
        "Content"
    ]

    normalized_columns = {
        str(col).strip().lower(): col
        for col in df.columns
    }

    for col in possible_columns:

        if col.lower() in normalized_columns:
            return normalized_columns[col.lower()]

    return None

# ============================================================
# 5. Tourism Classification Function
# ============================================================

def classify_text(text):

    text = clean_text(text)

    tourism_scores = {}

    for tourism_type, keywords in tourism_keywords.items():

        score = 0

        for word in keywords:

            word = clean_text(word)

            if word in text:
                score += 0.20

        tourism_scores[tourism_type] = round(min(score, 1.0), 2)

    # التصنيف الأساسي
    best_type = max(tourism_scores, key=tourism_scores.get)
    best_score = tourism_scores[best_type]

    # التصنيف الثانوي
    sorted_scores = sorted(
        tourism_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    secondary_type = "None"

    if len(sorted_scores) > 1 and sorted_scores[1][1] > 0:
        secondary_type = sorted_scores[1][0]

    # إذا لا يوجد تطابق
    if best_score == 0:
        best_type = "Other"

    return (
        best_type,
        secondary_type,
        best_score
    )

# ============================================================
# 6. Read All Excel Files
# ============================================================

files = glob.glob(input_path + r"\**\*.xlsx", recursive=True)

files = [
    f for f in files
    if not os.path.basename(f).startswith("~$")
]

print(f"Total Excel Files Found: {len(files)}")

# ============================================================
# 7. Process Files
# ============================================================

summary = []
errors = []

for i, file in enumerate(files, start=1):

    print(f"\nProcessing File {i}/{len(files)}")
    print(file)

    try:

        df = pd.read_excel(file, engine="openpyxl")

        if df.empty:
            continue

        # ====================================================
        # Extract Region from Folder Structure
        # ====================================================

        relative_path = os.path.relpath(file, input_path)

        path_parts = relative_path.split(os.sep)

        region = path_parts[0] if len(path_parts) > 0 else "Unknown"

        # ====================================================
        # Detect Text Column
        # ====================================================

        text_col = detect_text_column(df)

        if text_col is None:

            errors.append({
                "File": file,
                "Error": "No Text Column Found"
            })

            print("No Text Column Found")
            continue

        # ====================================================
        # Remove Old Columns if Exist
        # ====================================================

        old_columns = [
            "Tourism_Type",
            "Secondary_Tourism_Type",
            "Confidence_Score",
            "Region",
            "Classification_Source"
        ]

        for col in old_columns:
            if col in df.columns:
                df = df.drop(columns=[col])

        # ====================================================
        # Apply Classification
        # ====================================================

        tourism_types = []
        secondary_types = []
        confidence_scores = []

        for text in df[text_col]:

            tourism_type, secondary_type, confidence_score = classify_text(text)

            tourism_types.append(tourism_type)
            secondary_types.append(secondary_type)
            confidence_scores.append(confidence_score)

        # ====================================================
        # Add New Columns
        # ====================================================

        df["Region"] = region

        df["Tourism_Type"] = tourism_types

        df["Secondary_Tourism_Type"] = secondary_types

        df["Confidence_Score"] = confidence_scores

        df["Classification_Source"] = "Text-Based Classification"

        # ====================================================
        # Save File
        # ====================================================

        new_file_path = os.path.join(output_path, relative_path)

        os.makedirs(os.path.dirname(new_file_path), exist_ok=True)

        df.to_excel(new_file_path, index=False, engine="openpyxl")

        # ====================================================
        # Summary
        # ====================================================

        summary.append({
            "File_Name": os.path.basename(file),
            "Region": region,
            "Rows": len(df),
            "Text_Column": text_col,
            "Saved_To": new_file_path
        })

        print(f"Saved Successfully: {new_file_path}")

    except Exception as e:

        errors.append({
            "File": file,
            "Error": str(e),
            "Traceback": traceback.format_exc()
        })

        print("ERROR:")
        print(e)

# ============================================================
# 8. Save Summary and Errors
# ============================================================

summary_df = pd.DataFrame(summary)

errors_df = pd.DataFrame(errors)

summary_df.to_excel(summary_output, index=False, engine="openpyxl")

errors_df.to_excel(error_output, index=False, engine="openpyxl")

# ============================================================
# 9. Final Statistics
# ============================================================

print("\n====================================================")
print("TikTok Tourism Classification Completed Successfully")
print("====================================================")

print(f"Processed Files: {len(summary_df)}")

print(f"Error Files: {len(errors_df)}")

print(f"\nSummary File Saved At:")
print(summary_output)

print(f"\nError Log Saved At:")
print(error_output)

Total Excel Files Found: 598

Processing File 1/598
C:\Users\aws12\Desktop\Create Turism Type\Tik Tok Datasets - After Processing\المنطقة الجنوبية\بيانات منطقة الباحة - بعد المعالجة\Video comments 10_textready_analysis.xlsx
Saved Successfully: C:\Users\aws12\Desktop\Create Turism Type\OutPut Result\المنطقة الجنوبية\بيانات منطقة الباحة - بعد المعالجة\Video comments 10_textready_analysis.xlsx

Processing File 2/598
C:\Users\aws12\Desktop\Create Turism Type\Tik Tok Datasets - After Processing\المنطقة الجنوبية\بيانات منطقة الباحة - بعد المعالجة\Video comments 10_textready_cleaned.xlsx
Saved Successfully: C:\Users\aws12\Desktop\Create Turism Type\OutPut Result\المنطقة الجنوبية\بيانات منطقة الباحة - بعد المعالجة\Video comments 10_textready_cleaned.xlsx

Processing File 3/598
C:\Users\aws12\Desktop\Create Turism Type\Tik Tok Datasets - After Processing\المنطقة الجنوبية\بيانات منطقة الباحة - بعد المعالجة\Video comments 11_textready_analysis.xlsx
Saved Successfully: C:\Users\aws12\Desktop\Creat

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\aws12\\Desktop\\Create Turism Type\\summary output'

## **YouTube - Turisim Type**

In [3]:
# ============================================================
# Add Tourism_Type to ALL YouTube Excel Files
# Final Stable Version
# ============================================================

import pandas as pd
import glob
import os
import re
import traceback

# ============================================================
# 1. Paths
# ============================================================

input_path = r"C:\Users\aws12\Desktop\Create Turism Type\YouTube Datasets - After Processing"

output_path = r"C:\Users\aws12\Desktop\Create Turism Type\YouTube - Output Result"

summary_output = r"C:\Users\aws12\Desktop\Create Turism Type\summary youtube data\Summary Output.xlsx"

error_log_output = r"C:\Users\aws12\Desktop\Create Turism Type\error youtube data\Error Output.xlsx"

os.makedirs(output_path, exist_ok=True)

# ============================================================
# 2. Tourism Keywords Dictionary
# ============================================================

tourism_keywords = {

    "Cultural Tourism": [
        'متحف', 'مكتبة', 'تراث', 'ثقافي', 'معرض',
        'الحرف اليدوية', 'فنون', 'ثقافة',
        'حضاري', 'تراثي'
    ],

    "Entertainment Tourism": [
        'حديقة', 'بوليفارد', 'ملاهي', 'فعاليات',
        'ترفيه', 'شاليهات', 'منتجع',
        'مهرجان', 'العاب', 'ألعاب'
    ],

    "Historical Tourism": [
        'قلعة', 'آثار', 'اثار', 'تاريخي',
        'قصر', 'حصن', 'قرية',
        'أثري', 'اثرية', 'تراثية'
    ],

    "Religious Tourism": [
        'مسجد', 'جامع', 'الحرم',
        'نبوي', 'الكعبة', 'مصلى',
        'ديني', 'اسلامي', 'إسلامي'
    ],

    "Nature Tourism": [
        'جبل', 'غابة', 'شلال', 'وادي',
        'شاطئ', 'بحر', 'بحيرة',
        'منتزه', 'أكواخ', 'اكواخ',
        'جزيرة', 'مزرعة', 'محمية',
        'موانئ', 'ميناء', 'مطل'
    ]
}

# ============================================================
# 3. Helper Functions
# ============================================================

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def normalize_col_name(col):

    return str(col).strip().lower().replace(" ", "").replace("_", "")


def find_column(df, possible_names):

    normalized_columns = {
        normalize_col_name(col): col
        for col in df.columns
    }

    for name in possible_names:

        key = normalize_col_name(name)

        if key in normalized_columns:
            return normalized_columns[key]

    return None


def classify_place(category, place_name, file_name):

    category = clean_text(category)
    place_name = clean_text(place_name)
    file_name = clean_text(file_name)

    best_type = "Other"
    best_score = 0

    for tourism_type, keywords in tourism_keywords.items():

        score = 0

        for word in keywords:

            word = clean_text(word)

            if word in category:
                score += 0.50

            if word in place_name:
                score += 0.30

            if word in file_name:
                score += 0.20

        if score > best_score:

            best_score = score
            best_type = tourism_type

    return best_type, round(min(best_score, 1.0), 2)

# ============================================================
# 4. Collect Excel Files
# ============================================================

files = glob.glob(input_path + r"\**\*.xlsx", recursive=True)

files = [
    f for f in files
    if not os.path.basename(f).startswith("~$")
]

print(f"Total Excel files found: {len(files)}")

# ============================================================
# 5. Process All Files
# ============================================================

summary = []
errors = []

for i, file in enumerate(files, start=1):

    print(f"\nProcessing {i}/{len(files)}")
    print(file)

    try:

        df = pd.read_excel(file, engine="openpyxl")

        if df.empty:

            errors.append({
                "File": file,
                "Error": "Empty file"
            })

            print("Skipped: Empty file")
            continue

        category_col = find_column(df, [
            "categoryName",
            "CategoryName"
        ])

        place_col = find_column(df, [
            "title",
            "Title",
            "Place Name",
            "PlaceName",
            "Place_Name"
        ])

        category = ""

        if category_col is not None:

            category_series = df[category_col].dropna()

            if len(category_series) > 0:
                category = category_series.iloc[0]

        place_name = ""

        if place_col is not None:

            place_series = df[place_col].dropna()

            if len(place_series) > 0:
                place_name = place_series.iloc[0]

        file_name = os.path.basename(file)

        tourism_type, confidence_score = classify_place(
            category,
            place_name,
            file_name
        )

        # حذف الأعمدة القديمة إذا موجودة
        old_columns = [
            "نوع السياحة",
            "درجة الثقة",
            "Tourism_Type",
            "Confidence_Score"
        ]

        for col in old_columns:

            if col in df.columns:
                df = df.drop(columns=[col])

        # إضافة الأعمدة الجديدة
        df["Tourism_Type"] = tourism_type
        df["Confidence_Score"] = confidence_score

        # حفظ بنفس هيكلة المجلدات
        relative_path = os.path.relpath(file, input_path)

        new_file_path = os.path.join(output_path, relative_path)

        os.makedirs(os.path.dirname(new_file_path), exist_ok=True)

        df.to_excel(new_file_path, index=False, engine="openpyxl")

        summary.append({
            "File_Name": file_name,
            "Place_Name": place_name,
            "CategoryName": category,
            "Tourism_Type": tourism_type,
            "Confidence_Score": confidence_score,
            "Saved_To": new_file_path
        })

        print(f"Saved: {file_name} -> {tourism_type}")

    except Exception as e:

        errors.append({
            "File": file,
            "Error": str(e),
            "Traceback": traceback.format_exc()
        })

        print("ERROR:")
        print(e)

# ============================================================
# 6. Save Summary and Error Log
# ============================================================

summary_df = pd.DataFrame(summary)
errors_df = pd.DataFrame(errors)

summary_df.to_excel(summary_output, index=False, engine="openpyxl")

errors_df.to_excel(error_log_output, index=False, engine="openpyxl")

print("\nDONE")
print(f"Processed files: {len(summary_df)}")
print(f"Error files: {len(errors_df)}")
print(f"Summary saved at: {summary_output}")
print(f"Error log saved at: {error_log_output}")

Total Excel files found: 408

Processing 1/408
C:\Users\aws12\Desktop\Create Turism Type\YouTube Datasets - After Processing\المنطقة الجنوبية\بيانات اليوتيوب لمنطقة الباحة - بعد المعالجة\Al-Baha Vedio Comments 10_textready_analysis.xlsx
Saved: Al-Baha Vedio Comments 10_textready_analysis.xlsx -> Nature Tourism

Processing 2/408
C:\Users\aws12\Desktop\Create Turism Type\YouTube Datasets - After Processing\المنطقة الجنوبية\بيانات اليوتيوب لمنطقة الباحة - بعد المعالجة\Al-Baha Vedio Comments 10_textready_cleaned.xlsx
Saved: Al-Baha Vedio Comments 10_textready_cleaned.xlsx -> Nature Tourism

Processing 3/408
C:\Users\aws12\Desktop\Create Turism Type\YouTube Datasets - After Processing\المنطقة الجنوبية\بيانات اليوتيوب لمنطقة الباحة - بعد المعالجة\Al-Baha Vedio Comments 11_textready_analysis.xlsx
Saved: Al-Baha Vedio Comments 11_textready_analysis.xlsx -> Nature Tourism

Processing 4/408
C:\Users\aws12\Desktop\Create Turism Type\YouTube Datasets - After Processing\المنطقة الجنوبية\بيانات اليو